# Experiment 38 — PheromoneWalker + eligibility traces

A **surgical extension of the simple from-scratch PheromoneWalker v1** that reached ~0.00235 validation NDCG@10 on Beauty.

Everything from v1 stays fixed: corrected SparseWalker v1.1 recurrence, unique item→concept identity geometry, target-concept reward, evaporation, slow epoch-level rewiring, no warm start, no optimizer, no backward/autograd learning.

The only new mechanism is a bounded per-user eligibility trail: a current next-item outcome can reinforce edges used in the preceding few events with geometrically decayed credit.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, shutil, subprocess, json, runpy, torch
from pathlib import Path

REPO='/content/Sparsewalker'
BRANCH='agent/pheromone-eligibility-v3'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO], check=True)
for p in [f'{REPO}/src', f'{REPO}/experiments', f'{REPO}/benchmarks']:
    if p not in sys.path: sys.path.insert(0,p)
import sparsewalker
assert torch.cuda.is_available(), 'GPU runtime required'
print('GPU',torch.cuda.get_device_name(0),'torch',torch.__version__)
print('BRANCH',BRANCH,'PACKAGE',sparsewalker.__file__)


## Run

Defaults are intentionally moderate: 4-event eligibility horizon, decay 0.60, gain 0.50. The direct v1 reward is still present at full strength.


In [ ]:
SCRIPT=f'{REPO}/experiments/run_amazon_pheromone_eligibility.py'
sys.argv=[SCRIPT,
    '--dataset','beauty',
    '--epochs','12',
    '--batch-size','512',
    '--eligibility-steps','4',
    '--eligibility-decay','0.60',
    '--eligibility-gain','0.50',
]
runpy.run_path(SCRIPT, run_name='__main__')


## Inspect trajectory

The main falsification target is **best val NDCG@10 > 0.00235** (simple PheromoneWalker v1). Also watch whether `mean_trace_reward` and validation NDCG move together rather than diverging.


In [ ]:
import pandas as pd
root=Path('/content/drive/MyDrive/sparsewalker_pheromone_eligibility_v3/beauty/seed42')
h=root/'history.json'
if h.exists():
    df=pd.DataFrame(json.loads(h.read_text()))
    cols=['epoch','val_NDCG@10','val_HR@10','mean_edge_reward','mean_trace_reward','trace_credit_fraction','rewired_edges','tau_mean','tau_max','positions_per_s']
    display(df[[c for c in cols if c in df.columns]])
    best=df.loc[df['val_NDCG@10'].idxmax()]
    print('BEST EPOCH',int(best.epoch),'VAL NDCG@10',float(best['val_NDCG@10']))
    print('VS SIMPLE PHEROMONE V1',float(best['val_NDCG@10'])/0.0023507147682577624)
r=root/'result.json'
if r.exists():
    print(json.dumps(json.loads(r.read_text()),indent=2))


## Interpretation

- **> 0.00235 val NDCG@10:** delayed forward credit adds value beyond immediate ACO reward.
- Similar to v1: eligibility is unnecessary for these short Amazon sequences.
- Worse than v1: delayed credit is smearing the otherwise useful local signal; shorten horizon or reduce gain rather than adding more mechanisms.
- This experiment remains fully backward-free and from scratch.
